# Transformer EN→DE 학습 (Google Colab)

**런타임 설정**: 상단 메뉴 → 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후 실행

## 1. GPU 확인

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

## 2. 저장소 클론 및 의존성 설치

In [ ]:
import os

if not os.path.exists('/content/MyTransformer'):
    !git clone https://github.com/youngpark-POS/MyTransformer.git
else:
    print('이미 클론됨, 최신 코드로 업데이트...')
    !git -C /content/MyTransformer pull

%cd /content/MyTransformer

In [ ]:
import os
# subprocess(!python ...)가 src 레이아웃 패키지를 찾을 수 있도록 PYTHONPATH 설정
# NOTE: pip install -e . 의 editable finder는 Colab에서 transformer 서브패키지(data/model)를
#       노출하지 못해 ModuleNotFoundError를 일으키므로 사용하지 않고 PYTHONPATH만 쓴다.
os.environ['PYTHONPATH'] = '/content/MyTransformer/src'

# 혹시 이전 실행에서 남은 editable install이 PYTHONPATH를 가로채지 않도록 제거(있을 때만)
!pip uninstall -y transformer -q 2>/dev/null

# torch는 Colab에 기설치 — datasets, sacrebleu, tqdm만 추가 설치
!pip install -q datasets sacrebleu tqdm

## 3. 학습

In [ ]:
# 첫 실행 시 HuggingFace에서 Multi30k 다운로드 + vocab 캐시 생성 (수 분 소요)
!python -m transformer.train --epochs 10

## 4. 체크포인트를 Google Drive에 저장

> Colab 세션이 끊기면 `/content` 내용이 사라집니다. Drive에 백업해두세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil, os
dst = '/content/drive/MyDrive/MyTransformer'
os.makedirs(dst, exist_ok=True)
shutil.copy('checkpoints/best.pt', dst + '/best.pt')
print('저장 완료:', dst + '/best.pt')

## 5. 번역 추론

In [ ]:
!python -m transformer.translate --text "a man is riding a bike"

In [ ]:
# beam search (beam=5)
!python -m transformer.translate --text "a man is riding a bike" --beam 5

## 6. (선택) Drive에서 체크포인트 불러와 추론

세션이 재시작된 경우, Drive에서 체크포인트를 복원해 추론만 실행합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
os.makedirs('checkpoints', exist_ok=True)
shutil.copy('/content/drive/MyDrive/MyTransformer/best.pt', 'checkpoints/best.pt')
print('체크포인트 복원 완료')

In [ ]:
!python -m transformer.translate --text "two dogs are playing in the park" --beam 5